In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV 
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder 
from sklearn.pipeline import Pipeline 
from scikeras.wrappers import KerasClassifier
import tensorflow as tf  
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense  
from tensorflow.keras.callbacks import EarlyStopping  
import pickle 

In [40]:
data=pd.read_csv("Churn_Modelling.csv")

In [41]:
data=data.drop(columns=["RowNumber","CustomerId","Surname"],axis=1,inplace=False)
## Encode categorical variables 

label_encoder=LabelEncoder()
data['Gender']=label_encoder.fit_transform(data['Gender'])


## One-hot encode 'Geography' column

one_hot_encoded_geo=OneHotEncoder(handle_unknown='ignore')

geo_encoded=one_hot_encoded_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=one_hot_encoded_geo.get_feature_names_out(['Geography']))

In [42]:
geo_encoded

## One-hot encode 'Geography' column

one_hot_encoded_geo=OneHotEncoder(handle_unknown='ignore')

geo_encoded=one_hot_encoded_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=one_hot_encoded_geo.get_feature_names_out(['Geography']))

In [43]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [44]:

## split the data into features and target 
X = data.drop('Exited', axis=1)
y = data['Exited']

In [45]:
## split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

X_train_scaled shape: (8000, 12)
X_test_scaled shape: (2000, 12)


In [46]:
## Save the encoders and scaler for future use
with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

with open('onehot_encoder_geo.pkl', 'wb') as f:
    pickle.dump(one_hot_encoded_geo, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Encoders and scaler saved successfully!")

Encoders and scaler saved successfully!


In [47]:
## Define a function to create the model with configurable architecture
def create_model(optimizer='adam', layers=1, neurons=32):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train_scaled.shape[1],)))
    
    for _ in range(layers - 1):
        model.add(Dense(neurons, activation='relu'))
    
    # output layer for binary classification
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# test the model creation
test_model = create_model()
print(test_model.summary())

/Users/anupdangi/Desktop/AnupAI/Research/DS_ML_DL_NLP_BOOTCAMP/venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 449 (1.75 KB)

 Trainable params: 449 (1.75 KB)

 Non-trainable params: 0 (0.00 B)

None


In [48]:
## Create a Keras classifier wrapper
model = KerasClassifier(
    model=create_model,
    epochs=50,
    batch_size=10,
    verbose=0
)

print("KerasClassifier created successfully!")

KerasClassifier created successfully!


In [49]:
## parameter grid for hyperparameter tuning
# with scikeras KerasClassifier the arguments passed to the model function
# must be prefixed with `model__` in the grid
param_grid = {
    'model__neurons': [16, 32, 64],
    'model__layers': [1, 2],
    'epochs': [50, 100]
}

print("Parameter grid defined successfully!")

Parameter grid defined successfully!


In [50]:
## Perform grid search with cross-validation
# note: X_train_scaled was created earlier; ensure features match
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, verbose=1)
# fitting will search across the grid and store the result
grid_result = grid.fit(X_train_scaled, y_train)

# print the best parameters and score
print(f"\nBest accuracy: {grid_result.best_score_:.4f}")
print(f"Best parameters: {grid_result.best_params_}")

Fitting 3 folds for each of 12 candidates, totalling 36 fits


ValueError: Invalid parameter layers for estimator KerasClassifier.
This issue can likely be resolved by setting this parameter in the KerasClassifier constructor:
`KerasClassifier(layers=1)`
Check the list of available parameters with `estimator.get_params().keys()`

In [51]:
## Train final model with best parameters and save it
if 'grid_result' in globals():
    best_model = grid_result.best_estimator_

    # evaluate on test set
    test_score = best_model.score(X_test_scaled, y_test)
    print(f"Test Accuracy: {test_score:.4f}")

    # get the underlying Keras model to save
    final_keras_model = best_model.model
    final_keras_model.save('model.h5')
    print("Model saved to model.h5")

    # make predictions on test set
    y_pred = best_model.predict(X_test_scaled)
    print(f"\nSample predictions (first 10): {y_pred[:10]}")
    print(f"Actual labels (first 10): {y_test.values[:10]}")
else:
    print("Grid search did not complete; cannot train final model.")
else:

NameError: name 'grid_result' is not defined

In [ ]:
# demonstration: load the saved model and predict on a sample
from tensorflow.keras.models import load_model

try:
    demo_model = load_model('model.h5')
    sample = X_test_scaled[0].reshape(1, -1)
    prob = demo_model.predict(sample)[0][0]
    print(f"Sample prediction probability: {prob:.4f}")
    print(f"Actual label: {y_test.values[0]}")
except Exception as e:
    print('Demo model not available or error:', e)
